OIBSIP Data Analytics Internship

Task 3: Cleaning Data

Objective

To identify and fix data-quality issues in the cafe sales dataset and prepare a clean dataset for analysis.

Tools

Python, pandas, NumPy, Matplotlib, Seaborn, Jupyter Notebook

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df_raw = pd.read_csv("dirty_cafe_sales.csv")
df = df_raw.copy()

print("Dataset loaded successfully")
print("Rows and Columns", df.shape)
display(df.head())

Dataset loaded successfully
Rows and Columns (10000, 8)


,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2,4,Credit Card,Takeaway,08-09-2023
1,TXN_4977031,Cake,4,3,12,Cash,In-store,16-05-2023
2,TXN_4271903,Cookie,4,1,ERROR,Credit Card,In-store,19-07-2023
3,TXN_7034554,Salad,2,5,10,UNKNOWN,UNKNOWN,27-04-2023
4,TXN_3160411,Coffee,2,2,4,Digital Wallet,In-store,11-06-2023


### 1. Data Quality Audit
We will scan the dataset for explicit nulls (`NaN`) and hidden text placeholders like `ERROR` and `UNKNOWN` to understand exactly how many cells are corrupted before cleaning.

In [15]:
sentinels = ["ERROR", "UNKNOWN"]
df_check = df_raw.replace(
    sentinels, np.nan
)
quality_report = pd.DataFrame({
    "Data type": df_raw.dtypes.astype(str),
    "Missing/Corrupt Values": df_check.isna().sum(),
    "Missing Percentage": (
        df_check.isna().mean() * 100
    ).round(2),
    "Unique Values": df_raw.nunique()
})

print("Data Quality Report")
display(quality_report)

print("Duplicate Rows: ", df_raw.duplicated().sum())
display(df_raw.describe(include="all").T)

Data Quality Report


,Data type,Missing/Corrupt Values,Missing Percentage,Unique Values
Transaction ID,str,0,0.00,10000
Item,str,969,9.69,10
Quantity,str,479,4.79,7
Price Per Unit,str,533,5.33,8
Total Spent,str,502,5.02,19
Payment Method,str,3178,31.78,5
Location,str,3961,39.61,4
Transaction Date,str,460,4.60,367


Duplicate Rows:  0


,count,unique,top,freq
Transaction ID,10000,10000,TXN_1961373,1
Item,9667,10,Juice,1171
Quantity,9862,7,5,2013
Price Per Unit,9821,8,3,2429
Total Spent,9827,19,6,979
Payment Method,7421,5,Digital Wallet,2291
Location,6735,4,Takeaway,3022
Transaction Date,9841,367,UNKNOWN,159


## Initial Data Quality Assessment

### Dataset Overview
- Total rows: 10,000
- Total columns: 8
- Duplicate rows: 0
- Transaction IDs are unique.

### Identified Issues
1. Missing or corrupt values occur in multiple columns.
2. Payment Method has 3,178 missing/corrupt values (31.78%).
3. Location has 3,961 missing/corrupt values (39.61%).
4. Numeric columns contain text placeholders such as ERROR
   and require numeric conversion.
5. Transaction Date contains UNKNOWN and other invalid or
   missing values that require date conversion.
6. No exact duplicate rows were identified.

### Next Steps
- Standardize placeholder values.
- Correct data types.
- Investigate missing values and unusual numeric values.
- Document each cleaning decision.

In [7]:
numeric_colm = ["Quantity", "Price Per Unit", "Total Spent"]
for col in numeric_colm:
    values = pd.to_numeric(df_raw[col], errors = "coerce")
    print(f"\n---{col}---")
    print("Minimum: ", values.min())
    print("Maximum: ", values.max())
    print("Negative Values: ", (values < 0).sum())
    print("Missing / Invalid Values: ", values.isna().sum())


---Quantity---
Minimum:  1.0
Maximum:  5.0
Negative Values:  0
Missing / Invalid Values:  479

---Price Per Unit---
Minimum:  1.0
Maximum:  5.0
Negative Values:  0
Missing / Invalid Values:  533

---Total Spent---
Minimum:  1.0
Maximum:  25.0
Negative Values:  0
Missing / Invalid Values:  502


In [8]:
for col in numeric_colm:
    print(f"\nRows with invalid values in {col}")
    mask = pd.to_numeric(df_raw[col], errors = "coerce").isna()
    display(df_raw.loc[mask].head(5))


Rows with invalid values in Quantity


,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
20,TXN_3522028,Smoothie,ERROR,4,20,Cash,In-store,04-04-2023
55,TXN_5522862,Cookie,ERROR,1,2,Credit Card,Takeaway,19-03-2023
57,TXN_2080895,Cake,UNKNOWN,3,3,Digital Wallet,In-store,19-04-2023
66,TXN_8501819,Juice,NaN,3,6,Cash,NaN,30-03-2023
117,TXN_2148617,Juice,ERROR,3,9,Digital Wallet,UNKNOWN,10-01-2023



Rows with invalid values in Price Per Unit


,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
56,TXN_3578141,Cake,5,NaN,15,NaN,Takeaway,27-06-2023
65,TXN_4987129,Sandwich,3,NaN,NaN,NaN,In-store,20-10-2023
68,TXN_8427104,Salad,2,ERROR,10,NaN,In-store,27-10-2023
85,TXN_8035512,Tea,3,NaN,4.5,Cash,UNKNOWN,29-10-2023
104,TXN_7447872,Juice,2,NaN,6,NaN,NaN,NaN



Rows with invalid values in Total Spent


,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
2,TXN_4271903,Cookie,4,1,ERROR,Credit Card,In-store,19-07-2023
25,TXN_7958992,Smoothie,3,4,NaN,UNKNOWN,UNKNOWN,13-12-2023
31,TXN_8927252,UNKNOWN,2,1,ERROR,Credit Card,ERROR,06-11-2023
42,TXN_6650263,Tea,2,1.5,UNKNOWN,NaN,Takeaway,10-01-2023
65,TXN_4987129,Sandwich,3,NaN,NaN,NaN,In-store,20-10-2023


In [9]:
print("Most common transaction dates: ")
display(df_raw["Transaction Date"].value_counts().head(10))

print("\n Examples of date values: ")
display(df_raw["Transaction Date"].sample(10, random_state = 42))

Most common transaction dates: 


Transaction Date
UNKNOWN       159
ERROR         142
06-02-2023     40
16-06-2023     40
13-03-2023     39
21-07-2023     39
24-07-2023     39
21-09-2023     39
05-01-2023     38
18-06-2023     37
Name: count, dtype: int64


 Examples of date values: 


6252    17-02-2023
4684    16-07-2023
1731    12-09-2023
4742    02-04-2023
4521    30-09-2023
6340    03-10-2023
576     09-04-2023
5202    18-02-2023
6363    18-12-2023
439     31-07-2023
Name: Transaction Date, dtype: str

In [8]:

df_clean = df.copy()

# Step 1: Replace invalid text values in all columns
df_clean = df_clean.replace(
    ["ERROR", "UNKNOWN"], np.nan
)

# Step 2: Convert numeric columns
for col in numeric_colm:
    df_clean[col] = pd.to_numeric(
        df_clean[col], errors="coerce"
    )

# Step 3: Convert dates to datetime format
df_clean["Transaction Date"] = pd.to_datetime(
    df_clean["Transaction Date"],
    format="%d-%m-%Y",
    errors="coerce"
)

# Step 4: Display data types
print("Data types after conversion:")
display(df_clean.dtypes)

# Step 5: Count missing values
print("Missing values after conversion:")
display(df_clean.isna().sum())


Data types after conversion:


Transaction ID                 str
Item                           str
Quantity                   float64
Price Per Unit             float64
Total Spent                float64
Payment Method                 str
Location                       str
Transaction Date    datetime64[us]
dtype: object

Missing values after conversion:


Transaction ID         0
Item                 969
Quantity             479
Price Per Unit       533
Total Spent          502
Payment Method      3178
Location            3961
Transaction Date     460
dtype: int64

In [16]:

# Check how many rows have missing values
print("Total rows:", len(df_clean))

print(
    "Rows with at least one missing value:",
    df_clean.isna().any(axis=1).sum()
)

print(
    "Rows with all values missing:",
    df_clean.isna().all(axis=1).sum()
)

# Display sample rows with missing values
print("\nRows with missing Item:")
display(df_clean[df_clean["Item"].isna()].head(5))

print("\nRows with missing numeric values:")
display(
    df_clean[
        df_clean[["Quantity", "Price Per Unit", "Total Spent"]]
        .isna().any(axis=1)
    ].head(10)
)

print("\nRows with missing Payment Method or Location:")
display(
    df_clean[
        df_clean[["Payment Method", "Location"]]
        .isna().any(axis=1)
    ].head(10)
)


Total rows: 10000
Rows with at least one missing value: 6911
Rows with all values missing: 0

Rows with missing Item:


,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
6,TXN_4433211,NaN,3.0,3.0,9.0,NaN,Takeaway,2023-10-06
8,TXN_4717867,NaN,5.0,3.0,15.0,NaN,Takeaway,2023-07-28
14,TXN_8915701,NaN,2.0,1.5,3.0,NaN,In-store,2023-03-21
30,TXN_1736287,NaN,5.0,2.0,10.0,Digital Wallet,NaN,2023-06-02
31,TXN_8927252,NaN,2.0,1.0,NaN,Credit Card,NaN,2023-11-06



Rows with missing numeric values:


,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
2,TXN_4271903,Cookie,4.0,1.0,NaN,Credit Card,In-store,2023-07-19
20,TXN_3522028,Smoothie,NaN,4.0,20.0,Cash,In-store,2023-04-04
25,TXN_7958992,Smoothie,3.0,4.0,NaN,NaN,NaN,2023-12-13
31,TXN_8927252,NaN,2.0,1.0,NaN,Credit Card,NaN,2023-11-06
42,TXN_6650263,Tea,2.0,1.5,NaN,NaN,Takeaway,2023-01-10
55,TXN_5522862,Cookie,NaN,1.0,2.0,Credit Card,Takeaway,2023-03-19
56,TXN_3578141,Cake,5.0,NaN,15.0,NaN,Takeaway,2023-06-27
57,TXN_2080895,Cake,NaN,3.0,3.0,Digital Wallet,In-store,2023-04-19
65,TXN_4987129,Sandwich,3.0,NaN,NaN,NaN,In-store,2023-10-20
66,TXN_8501819,Juice,NaN,3.0,6.0,Cash,NaN,2023-03-30



Rows with missing Payment Method or Location:


,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
3,TXN_7034554,Salad,2.0,5.0,10.0,NaN,NaN,2023-04-27
5,TXN_2602893,Smoothie,5.0,4.0,20.0,Credit Card,NaN,2023-03-31
6,TXN_4433211,NaN,3.0,3.0,9.0,NaN,Takeaway,2023-10-06
7,TXN_6699534,Sandwich,4.0,4.0,16.0,Cash,NaN,2023-10-28
8,TXN_4717867,NaN,5.0,3.0,15.0,NaN,Takeaway,2023-07-28
9,TXN_2064365,Sandwich,5.0,4.0,20.0,NaN,In-store,2023-12-31
13,TXN_9437049,Cookie,5.0,1.0,5.0,NaN,Takeaway,2023-06-01
14,TXN_8915701,NaN,2.0,1.5,3.0,NaN,In-store,2023-03-21
16,TXN_3765707,Sandwich,1.0,4.0,4.0,NaN,NaN,2023-06-10
18,TXN_8876618,Cake,5.0,3.0,15.0,Cash,NaN,2023-03-25


In [18]:
categorial_colm = ["Item",
                  "Payment Method",
                  "Location"]
for col in categorial_colm:
    df_clean[col] = df_clean[col].fillna("Unknown")

print("Missing values after categorial cleaning: ")
display(df_clean.isna().sum())

Missing values after categorial cleaning: 


Transaction ID        0
Item                  0
Quantity            479
Price Per Unit      533
Total Spent         502
Payment Method        0
Location              0
Transaction Date    460
dtype: int64

In [ ]:
Missing values after categorial cleaning: 
Transaction ID        0
Item                  0
Quantity            479
Price Per Unit      533
Total Spent         502
Payment Method        0
Location              0
Transaction Date    460
dtype: int64

In [19]:

# Check relationships between numeric columns
print("Quantity and Total Spent both missing:")
display(
    df_clean[
        df_clean["Quantity"].isna()
        & df_clean["Total Spent"].isna()
    ].head(10)
)

print("Price Per Unit and Total Spent both missing:")
display(
    df_clean[
        df_clean["Price Per Unit"].isna()
        & df_clean["Total Spent"].isna()
    ].head(10)
)

# Check whether complete numeric records follow
# Quantity × Price Per Unit = Total Spent
complete = df_clean[
    ["Quantity", "Price Per Unit", "Total Spent"]
].notna().all(axis=1)

expected_total = (
    df_clean.loc[complete, "Quantity"]
    * df_clean.loc[complete, "Price Per Unit"]
)

actual_total = df_clean.loc[complete, "Total Spent"]

print("Complete numeric records:", complete.sum())
print(
    "Records where total differs from quantity × price:",
    (~np.isclose(actual_total, expected_total)).sum()
)


Quantity and Total Spent both missing:


,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
236,TXN_8562645,Salad,NaN,5.0,NaN,Unknown,In-store,2023-05-18
278,TXN_3229409,Juice,NaN,3.0,NaN,Cash,Takeaway,2023-04-15
641,TXN_2962976,Juice,NaN,3.0,NaN,Unknown,Unknown,2023-03-17
738,TXN_8696094,Sandwich,NaN,4.0,NaN,Unknown,Takeaway,2023-05-14
2796,TXN_9188692,Cake,NaN,3.0,NaN,Credit Card,Unknown,2023-12-01
3203,TXN_4565754,Smoothie,NaN,4.0,NaN,Digital Wallet,Takeaway,2023-10-06
3224,TXN_6297232,Coffee,NaN,2.0,NaN,Unknown,Unknown,2023-04-07
3401,TXN_3251829,Tea,NaN,1.5,NaN,Digital Wallet,In-store,2023-07-25
4257,TXN_6470865,Coffee,NaN,2.0,NaN,Digital Wallet,Takeaway,2023-09-18
5841,TXN_5884081,Cookie,NaN,1.0,NaN,Digital Wallet,In-store,2023-07-05


Price Per Unit and Total Spent both missing:


,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
65,TXN_4987129,Sandwich,3.0,NaN,NaN,Unknown,In-store,2023-10-20
1674,TXN_9367492,Tea,2.0,NaN,NaN,Cash,In-store,2023-06-19
1761,TXN_3611851,Unknown,4.0,NaN,NaN,Credit Card,Unknown,2023-02-09
2229,TXN_8498613,Sandwich,2.0,NaN,NaN,Unknown,Unknown,2023-11-08
2289,TXN_7524977,Unknown,4.0,NaN,NaN,Unknown,Unknown,2023-12-09
2585,TXN_1259340,Tea,3.0,NaN,NaN,Digital Wallet,Unknown,2023-02-24
3162,TXN_3577949,Cake,3.0,NaN,NaN,Unknown,Takeaway,2023-04-25
3598,TXN_2857444,Smoothie,1.0,NaN,NaN,Cash,Takeaway,2023-05-10
3673,TXN_6500126,Smoothie,2.0,NaN,NaN,Unknown,Unknown,NaT
4021,TXN_6424202,Cookie,2.0,NaN,NaN,Credit Card,In-store,2023-11-20


Complete numeric records: 8544
Records where total differs from quantity × price: 0


In [11]:
# Filling Total Spent where Quantity and Price per Unit are available
mask = (
    df_clean["Total Spent"].isna()
    & df_clean["Quantity"].notna()
    & df_clean["Price Per Unit"].notna()
)

df_clean.loc[mask, "Total Spent"] = (
    df_clean.loc[mask, "Quantity"] 
    * df_clean.loc[mask, "Price Per Unit"]
)
print("Total Spent values recovered: ", mask.sum())
print("Missing Total spent values remaining: ", df_clean["Total Spent"].isna().sum())

Total Spent values recovered:  462
Missing Total spent values remaining:  40


In [13]:
mask_quantity = (df_clean["Quantity"].isna()
        & df_clean["Total Spent"].notna()
        & df_clean["Price Per Unit"].notna()
        & (df_clean["Price Per Unit"] > 0)
       )
df_clean.loc[mask_quantity, "Quantity"] = (
    df_clean.loc[mask_quantity, "Total Spent"]
    / df_clean.loc[mask_quantity, "Price Per Unit"]
)

print("Quantity values recovered", mask_quantity.sum())
print("Missing Quantity values remaining: ", df_clean["Quantity"].isna().sum())

Quantity values recovered 441
Missing Quantity values remaining:  38


In [14]:
# Check whether recovered Quantity values are whole numbers
recovered = df_clean.loc[
    df_clean['Quantity'].notna() &
    df_clean['Quantity'].index.isin(
        df_clean.index[df_clean['Quantity'].notna()]
    ),
    'Quantity'
]

print("Non-whole Quantity values:")
print(recovered[recovered % 1 != 0].count())

Non-whole Quantity values:
0


In [15]:
recoverable_price = (
    df_clean['Price Per Unit'].isna()
    & df_clean['Total Spent'].notna()
    & df_clean['Quantity'].notna()
    & (df_clean['Quantity'] > 0)
)

# Calculate and fill the missing prices
df_clean.loc[recoverable_price, 'Price Per Unit'] = (
    df_clean.loc[recoverable_price, 'Total Spent']
    / df_clean.loc[recoverable_price, 'Quantity']
)

# Display the results
print("Price Per Unit values recovered:", recoverable_price.sum())
print(
    "Missing Price Per Unit values remaining:",
    df_clean['Price Per Unit'].isna().sum()
)

Price Per Unit values recovered: 495
Missing Price Per Unit values remaining: 38


In [16]:

# Step 5 — Validate Total Spent calculations

valid_rows = (
    df_clean['Quantity'].notna()
    & df_clean['Price Per Unit'].notna()
    & df_clean['Total Spent'].notna()
)

calculated_total = (
    df_clean.loc[valid_rows, 'Quantity']
    * df_clean.loc[valid_rows, 'Price Per Unit']
)

difference = (
    df_clean.loc[valid_rows, 'Total Spent']
    - calculated_total
).abs()

print("Rows checked:", valid_rows.sum())
print("Rows with mismatched totals:", (difference > 0.01).sum())


Rows checked: 9942
Rows with mismatched totals: 0


In [18]:

# Inspect all rows with missing numerical values

missing_rows = df_clean[
    df_clean[['Quantity', 'Price Per Unit', 'Total Spent']]
    .isna()
    .any(axis=1)
]

print("Rows with at least one missing numerical value:", len(missing_rows))

display(missing_rows)


Rows with at least one missing numerical value: 58


,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
65,TXN_4987129,Sandwich,3.0,NaN,NaN,NaN,In-store,2023-10-20
236,TXN_8562645,Salad,NaN,5.0,NaN,NaN,In-store,2023-05-18
278,TXN_3229409,Juice,NaN,3.0,NaN,Cash,Takeaway,2023-04-15
629,TXN_9289174,Cake,NaN,NaN,12.0,Digital Wallet,In-store,2023-12-30
641,TXN_2962976,Juice,NaN,3.0,NaN,NaN,NaN,2023-03-17
738,TXN_8696094,Sandwich,NaN,4.0,NaN,NaN,Takeaway,2023-05-14
912,TXN_1575608,Sandwich,NaN,NaN,20.0,NaN,Takeaway,2023-01-05
1008,TXN_7225428,Tea,NaN,NaN,3.0,Credit Card,Takeaway,2023-03-07
1436,TXN_7590801,Tea,NaN,NaN,6.0,Cash,Takeaway,NaT
1482,TXN_3593060,Smoothie,NaN,NaN,16.0,Cash,NaN,2023-03-05


In [19]:

# Step 7 — Count missing numerical values per row

numeric_cols = ['Quantity', 'Price Per Unit', 'Total Spent']

missing_count = df_clean[numeric_cols].isna().sum(axis=1)

print("Rows with 1 missing numerical value:", (missing_count == 1).sum())
print("Rows with 2 missing numerical values:", (missing_count == 2).sum())
print("Rows with 3 missing numerical values:", (missing_count == 3).sum())
print("Total rows with missing numerical values:", (missing_count > 0).sum())


Rows with 1 missing numerical value: 0
Rows with 2 missing numerical values: 58
Rows with 3 missing numerical values: 0
Total rows with missing numerical values: 58


In [20]:

# Step 8 — Remove rows with incomplete numerical data

rows_before = len(df_clean)

# Preserve a copy before removing rows
df_before_incomplete_removal = df_clean.copy()

# Remove rows where any of the three numerical columns is missing
df_clean = df_clean.dropna(
    subset=['Quantity', 'Price Per Unit', 'Total Spent']
).copy()

rows_after = len(df_clean)
rows_removed = rows_before - rows_after

print("Rows before removal:", rows_before)
print("Rows removed:", rows_removed)
print("Rows remaining:", rows_after)
print("Missing Quantity:", df_clean['Quantity'].isna().sum())
print("Missing Price Per Unit:", df_clean['Price Per Unit'].isna().sum())
print("Missing Total Spent:", df_clean['Total Spent'].isna().sum())


Rows before removal: 10000
Rows removed: 58
Rows remaining: 9942
Missing Quantity: 0
Missing Price Per Unit: 0
Missing Total Spent: 0


### Handling Remaining Missing Numerical Values

After attempting to recover missing values using the relationship between Quantity, Price Per Unit, and Total Spent, 58 rows still contained two missing numerical values each. These values could not be reliably recovered using the available information.

Therefore, the 58 rows were removed because the missing numerical information was insufficient for reliable sales calculations.

- Rows before removal: 10,000
- Rows removed: 58
- Rows remaining: 9,942
- Missing values remaining in Quantity, Price Per Unit, and Total Spent: 0

In [22]:
duplicated_count = df_clean.duplicated().sum()
print("Duplicate rows found", duplicated_count)

Duplicate rows found 0


### Duplicate Identification and Removal

Duplicate rows were identified using pandas `duplicated()` method. The dataset contained 0 duplicate rows after handling missing numerical values. Therefore, no additional rows were removed during duplicate handling.


In [26]:

 #Standardize text columns

text_cols = ['Item', 'Payment Method', 'Location']

for col in text_cols:
    df_clean[col] = df_clean[col].str.strip()

# Standardize Location spelling consistently
df_clean['Location'] = df_clean['Location'].replace({
    'In-store': 'In-Store',
    'Takeaway': 'Takeaway'
})

# Check the final unique values
for col in text_cols:
    print(f"\n{col}:")
    print(df_clean[col].value_counts(dropna=False))



Item:
Item
Juice       1167
Coffee      1158
Salad       1144
Cake        1134
Sandwich    1122
Smoothie    1089
Cookie      1084
Tea         1081
NaN          963
Name: count, dtype: int64

Payment Method:
Payment Method
NaN               3158
Digital Wallet    2280
Credit Card       2260
Cash              2244
Name: count, dtype: int64

Location:
Location
NaN         3940
Takeaway    3004
In-Store    2998
Name: count, dtype: int64


In [29]:

# Step 15 — Parse dates using the correct format

# Replace invalid date markers with missing values
original_dates = df_raw.loc[
    df_clean.index, 'Transaction Date'
].replace(['ERROR', 'UNKNOWN'], np.nan)

# Parse dates in DD-MM-YYYY format
df_clean['Transaction Date'] = pd.to_datetime(
    original_dates,
    format='%d-%m-%Y',
    errors='coerce'
)

# Check the results
print(
    "Missing or invalid dates remaining:",
    df_clean['Transaction Date'].isna().sum()
)

print("\nSample parsed dates:")
print(df_clean['Transaction Date'].dropna().head(10))


Missing or invalid dates remaining: 457

Sample parsed dates:
0   2023-09-08
1   2023-05-16
2   2023-07-19
3   2023-04-27
4   2023-06-11
5   2023-03-31
6   2023-10-06
7   2023-10-28
8   2023-07-28
9   2023-12-31
Name: Transaction Date, dtype: datetime64[us]


### Transaction Date Cleaning

The Transaction Date column was converted to datetime format using the `DD-MM-YYYY` format. Values marked as `ERROR`, `UNKNOWN`, or missing were converted to `NaT` (Not a Time).

After cleaning, 457 rows had missing or invalid transaction dates. These rows were retained because their other transaction details may still be useful for analysis. The missing dates will be documented as a limitation of the dataset.

In [31]:

#  Detect numerical outliers using IQR

numeric_cols = ['Quantity', 'Price Per Unit', 'Total Spent']

for col in numeric_cols:
    Q1 = df_clean[col].quantile(0.25)
    Q3 = df_clean[col].quantile(0.75)

    IQR = Q3 - Q1

    lower_limit = Q1 - 1.5 * IQR
    upper_limit = Q3 + 1.5 * IQR

    outliers = df_clean[
        (df_clean[col] < lower_limit) |
        (df_clean[col] > upper_limit)
    ]

    print(f"\nColumn: {col}")
    print("Q1:", Q1)
    print("Q3:", Q3)
    print("IQR:", IQR)
    print("Lower limit:", lower_limit)
    print("Upper limit:", upper_limit)
    print("Number of outliers:", len(outliers))



Column: Quantity
Q1: 2.0
Q3: 4.0
IQR: 2.0
Lower limit: -1.0
Upper limit: 7.0
Number of outliers: 0

Column: Price Per Unit
Q1: 2.0
Q3: 4.0
IQR: 2.0
Lower limit: -1.0
Upper limit: 7.0
Number of outliers: 0

Column: Total Spent
Q1: 4.0
Q3: 12.0
IQR: 8.0
Lower limit: -8.0
Upper limit: 24.0
Number of outliers: 267


In [32]:

# Step 19 — Inspect Total Spent outliers

Q1 = df_clean['Total Spent'].quantile(0.25)
Q3 = df_clean['Total Spent'].quantile(0.75)
IQR = Q3 - Q1

upper_limit = Q3 + 1.5 * IQR

outliers = df_clean[df_clean['Total Spent'] > upper_limit]

print("Number of outliers:", len(outliers))

print("\nHighest Total Spent values:")
print(
    outliers['Total Spent']
    .sort_values(ascending=False)
    .head(15)
)

print("\nOutlier transaction details:")
print(
    outliers[
        ['Transaction ID', 'Item', 'Quantity',
         'Price Per Unit', 'Total Spent']
    ].head(10)
)


Number of outliers: 267

Highest Total Spent values:
10     25.0
51     25.0
52     25.0
96     25.0
100    25.0
150    25.0
157    25.0
177    25.0
214    25.0
330    25.0
536    25.0
546    25.0
557    25.0
570    25.0
612    25.0
Name: Total Spent, dtype: float64

Outlier transaction details:
    Transaction ID   Item  Quantity  Price Per Unit  Total Spent
10     TXN_2548360  Salad       5.0             5.0         25.0
51     TXN_6342161  Salad       5.0             5.0         25.0
52     TXN_8914892    NaN       5.0             5.0         25.0
96     TXN_5220895  Salad       5.0             5.0         25.0
100    TXN_9517146    NaN       5.0             5.0         25.0
150    TXN_8687151  Salad       5.0             5.0         25.0
157    TXN_4283157  Salad       5.0             5.0         25.0
177    TXN_1896955  Salad       5.0             5.0         25.0
214    TXN_8693704  Salad       5.0             5.0         25.0
330    TXN_5523450  Salad       5.0             5.0  

### Outlier Detection and Treatment

The Interquartile Range (IQR) method was used to detect outliers in Quantity, Price Per Unit, and Total Spent.

- Quantity: 0 outliers detected.
- Price Per Unit: 0 outliers detected.
- Total Spent: 267 outliers detected using an upper limit of 24.0.

The flagged Total Spent values were inspected. The sample transactions were consistent with the relationship Total Spent = Quantity × Price Per Unit. Therefore, the outliers were retained because they may represent legitimate sales rather than data-entry errors. No values were capped or removed solely because of the IQR results.

In [34]:

# Convert Quantity to integer type

df_clean['Quantity'] = df_clean['Quantity'].astype('int64')

print(df_clean['Quantity'].dtype)


int64


In [35]:

#  Before-and-after comparison

print("BEFORE CLEANING")
print("----------------")
print("Rows:", df_raw.shape[0])
print("Columns:", df_raw.shape[1])
print("Duplicate rows:", df_raw.duplicated().sum())
print("Missing values per column:")
print(df_raw.isna().sum())

print("\nAFTER CLEANING")
print("---------------")
print("Rows:", df_clean.shape[0])
print("Columns:", df_clean.shape[1])
print("Duplicate rows:", df_clean.duplicated().sum())
print("Missing values per column:")
print(df_clean.isna().sum())

print("\nSUMMARY")
print("-------")
print("Rows removed:", df_raw.shape[0] - df_clean.shape[0])
print("Columns retained:", df_clean.shape[1])


BEFORE CLEANING
----------------
Rows: 10000
Columns: 8
Duplicate rows: 0
Missing values per column:
Transaction ID         0
Item                 333
Quantity             138
Price Per Unit       179
Total Spent          173
Payment Method      2579
Location            3265
Transaction Date     159
dtype: int64

AFTER CLEANING
---------------
Rows: 9942
Columns: 8
Duplicate rows: 0
Missing values per column:
Transaction ID         0
Item                 963
Quantity               0
Price Per Unit         0
Total Spent            0
Payment Method      3158
Location            3940
Transaction Date     457
dtype: int64

SUMMARY
-------
Rows removed: 58
Columns retained: 8


In [36]:

# Step 23 — Final Data Quality Report

quality_report = pd.DataFrame({
    'Original Data Type': df_raw.dtypes.astype(str),
    'Cleaned Data Type': df_clean.dtypes.astype(str),
    'Missing Before': df_raw.isna().sum(),
    'Missing After': df_clean.isna().sum()
})

quality_report['Missing After (%)'] = (
    quality_report['Missing After'] / len(df_clean) * 100
).round(2)

print("FINAL DATA QUALITY REPORT")
print("=========================")
display(quality_report)

print("\nDataset Summary")
print("---------------")
print("Original rows:", len(df_raw))
print("Cleaned rows:", len(df_clean))
print("Rows removed:", len(df_raw) - len(df_clean))
print("Columns retained:", len(df_clean.columns))
print("Duplicate rows remaining:", df_clean.duplicated().sum())
print("Total missing values remaining:", df_clean.isna().sum().sum())


FINAL DATA QUALITY REPORT


,Original Data Type,Cleaned Data Type,Missing Before,Missing After,Missing After (%)
Transaction ID,str,str,0,0,0.00
Item,str,str,333,963,9.69
Quantity,str,int64,138,0,0.00
Price Per Unit,str,float64,179,0,0.00
Total Spent,str,float64,173,0,0.00
Payment Method,str,str,2579,3158,31.76
Location,str,str,3265,3940,39.63
Transaction Date,str,datetime64[us],159,457,4.60



Dataset Summary
---------------
Original rows: 10000
Cleaned rows: 9942
Rows removed: 58
Columns retained: 8
Duplicate rows remaining: 0
Total missing values remaining: 8518


## Data Cleaning Summary

The cafe sales dataset originally contained 10,000 rows and 8 columns.

The following cleaning operations were performed:

1. **Missing and invalid values:** The markers `ERROR` and `UNKNOWN` were converted to missing values where appropriate.
2. **Numerical recovery:** Missing numerical values were recovered using the relationship between Quantity, Price Per Unit, and Total Spent wherever sufficient information was available.
3. **Incomplete numerical records:** 58 rows with unresolved missing numerical values were removed.
4. **Duplicate records:** No duplicate rows were found.
5. **Text standardization:** Leading and trailing spaces were removed from text columns, and Location labels were standardized.
6. **Date conversion:** Transaction Date was converted to datetime format using the `DD-MM-YYYY` format. Invalid and missing dates were retained as missing values.
7. **Outlier detection:** The IQR method identified 267 outliers in Total Spent. These were retained because the inspected sample values were consistent with the sales calculation.
8. **Data types:** Quantity was converted to integer format (`int64`).

### Final Result

The cleaned dataset contains 9,942 rows and 8 columns. Numerical transaction columns have no missing values, while some categorical fields and transaction dates still contain missing values. These values were retained rather than filled with invented information.

The cleaned dataset is ready for export and further analysis, subject to the documented limitations caused by missing categorical information and dates.

In [37]:

# Export cleaned dataset

df_clean.to_csv('cleaned_cafe_sales.csv', index=False)

print("Cleaned dataset exported successfully!")
print("File name: cleaned_cafe_sales.csv")


Cleaned dataset exported successfully!
File name: cleaned_cafe_sales.csv
